# Dalhousie university
## Price data exploration
#### Objective: Explore actual data to understand their changes over time, specially see where a price change could be a transitory or permanent change.

## 1. Loading and preparation of Data

### 1.1. Environment preparation

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


### 1.2. Data loading

In [ ]:
# load data using prjoect path
data = pd.read_csv(wd_d + "A1_clean.csv")

In [ ]:
# shows last ten (10) registers
data.tail(10)

### 1.3 Data preprocessing
We create new columns. Also split and clean data to product databases. 

In [ ]:
# erase null products
data = data[~data["precio"].isna()]

In [ ]:
# set date as index
data = data.set_index(data["fecha"])

In [ ]:
# drop duplicated data
data = data.drop_duplicates(["Descripcion", "fecha"])

In [ ]:
# create year, month, date columns
data['fecha'] = data['fecha'].astype(str)
data[["anio", "mes", "dia"]] = data["fecha"].str.split(pat = "-", expand = True)

In [ ]:
# change column type and weekday column
data["fecha"] = pd.to_datetime(data["fecha"])
data["weekday"] = data["fecha"].dt.day_name() # it would be important for analyzing weekend discounts and other seasonal discounts.

#### 1.3.1 Beer

In [ ]:
# filter data for only beer 
cervezadf = data[data["palabra"]=="cerveza"]

In [ ]:
# show last 10 register of beer data (data is dirty, it appear products like wine. Needs cleaning)
cervezadf.tail(10)

In [ ]:
# count unique varieties where liquid unit is null
cervezadf[cervezadf["unidad_liquida"].isna()].Descripcion.value_counts()

In [ ]:
# following the previous step, we erase that varieties
cervezadf = cervezadf[~cervezadf["unidad_liquida"].isna()]

In [ ]:
# looks for varieties that not has (cerv or pola or paulaner) in the description.
cervezadf[~cervezadf["Descripcion"].str.contains("cerv|pola|paulaner")].Descripcion.value_counts()

In [ ]:
# applied last filter followin printed varieties
cervezadf = cervezadf[cervezadf["Descripcion"].str.contains("cerv|pola|paulaner")]

In [ ]:
# look of actual data varieties
cervezadf.Descripcion.value_counts()

In [ ]:
# there are some noise yet, print this varieties
cervezadf[cervezadf["Descripcion"].str.contains("vaso|termo|pack ron")].Descripcion.value_counts()

In [ ]:
# applied last filter
cervezadf = cervezadf[~cervezadf["Descripcion"].str.contains("vaso|termo|pack ron|cold brew")]

In [ ]:
# look of actual data varieties again
cervezadf.Descripcion.value_counts()
# data looks clear

In [ ]:
# print data 
cervezadf.tail(10)

#### 1.3.2 Cigarettes

In [ ]:
# filter data for only cigarettes
cigarrillosdf = data[data["palabra"] == "cigarrillos"]

In [ ]:
# varieties in the actual data
cigarrillosdf.Descripcion.value_counts()

In [ ]:
# filter data when (cig or cgllo) are present in description
cigarrillosdf = cigarrillosdf[cigarrillosdf["Descripcion"].str.contains("cig|cgllo")]

In [ ]:
# count actual unique varieties
cigarrillosdf.Descripcion.value_counts()

In [ ]:
# following the previous step, we erase one noise variety
cigarrillosdf = cigarrillosdf[~cigarrillosdf["Descripcion"].str.contains("adaptador")]

## 2. Data analysis

Price analysis - most frequent product

In [ ]:
# products to filter
marcab = cervezadf.Descripcion.value_counts().keys()
marcac = cigarrillosdf.Descripcion.value_counts().keys()

In [ ]:
# dataframe with products info
marcadf = data[(data["Descripcion"].str.contains("|".join(marcab)))|(data["Descripcion"].str.contains('|'.join(marcac)))]
# select only interest columns of dataframe
marcadf = marcadf[["fecha", "Descripcion", "precio"]]
# reset index
marcadf = marcadf.reset_index(drop=True)

In [ ]:
# function to detect precio sequence
marcadf['fecha'] = pd.to_datetime(marcadf['fecha'])

# Sort the dataframe by product and date
marcadf = marcadf.sort_values(by=['Descripcion', 'fecha']).reset_index(drop=True)

# Create the first column: 'precio Type'
def precio_type(row, marcadf):
    product = row['Descripcion']
    date = row['fecha']
    
    # Check for yesterday and tomorrow
    yesterday = date - pd.Timedelta(days=1)
    tomorrow = date + pd.Timedelta(days=1)
    
    has_yesterday = marcadf[(marcadf['Descripcion'] == product) & (marcadf['fecha'] == yesterday)].shape[0] > 0
    has_tomorrow = marcadf[(marcadf['Descripcion'] == product) & (marcadf['fecha'] == tomorrow)].shape[0] > 0
    
    if not has_yesterday:
        return 'Initial price'
    elif has_yesterday and has_tomorrow:
        return 'Series price'
    elif has_yesterday and not has_tomorrow:
        return 'Final price'
    return np.nan

marcadf['Price Type'] = marcadf.apply(lambda row: precio_type(row, marcadf), axis=1)

# Create the second column: 'precio Change Type'
def precio_change_type(row, marcadf):
    product = row['Descripcion']
    date = row['fecha']
    current_precio = row['precio']
    
    # Check for yesterday and tomorrow
    yesterday = date - pd.Timedelta(days=1)
    tomorrow = date + pd.Timedelta(days=1)
    
    # Find the previous price
    previous_day = marcadf[(marcadf['Descripcion'] == product) & (marcadf['fecha'] < date)].sort_values(by='fecha', ascending=False).head(1)
    if not previous_day.empty:
        previous_precio = previous_day['precio'].values[0]
    else:
        return 'Constant'
    
    # If no change in precio
    if current_precio == previous_precio:
        return 'Constant'
    
    # Check if change maintains for the next two weeks
    next_two_weeks = marcadf[(marcadf['Descripcion'] == product) & (marcadf['fecha'] > date) & (marcadf['fecha'] <= date + pd.Timedelta(days=14))]
    if (next_two_weeks['precio'].nunique() == 1) and (next_two_weeks['precio'].values[0] == current_precio) and (next_two_weeks['fecha'].nunique() > 6):
        return 'Permanent change'
    else:
        return 'Transitory change'

marcadf['Price Change Type'] = marcadf.apply(lambda row: precio_change_type(row, marcadf), axis=1)

# Display the updated dataframe
# marcadf

In [ ]:
# Set seaborn style
sns.set(style="whitegrid")

# Function to visualize price changes with different markers and colors
def plot_product_price(product_df, product_name):
    plt.figure(figsize=(10, 6))

    # Plot the price over time
    sns.lineplot(data=product_df, x='fecha', y='precio', label=product_name, marker='', linewidth=2)

    # Mark 'Initial price' with a different marker
    initial_price = product_df[product_df['Price Type'] == 'Initial price']
    plt.scatter(initial_price['fecha'], initial_price['precio'], color='green', label='Initial price', marker='^', s=100)

    # Mark 'Final price' with a different marker
    final_price = product_df[product_df['Price Type'] == 'Final price']
    plt.scatter(final_price['fecha'], final_price['precio'], color='red', label='Final price', marker='v', s=100)

    # Highlight 'Transitory change' points
    transitory_change = product_df[product_df['Price Change Type'] == 'Transitory change']
    plt.scatter(transitory_change['fecha'], transitory_change['precio'], color='orange', label='Transitory change', marker='o', s=100)

    # Highlight 'Permanent change' points
    permanent_change = product_df[product_df['Price Change Type'] == 'Permanent change']
    plt.scatter(permanent_change['fecha'], permanent_change['precio'], color='purple', label='Permanent change', marker='o', s=100)

    # Customize the plot
    plt.title(f"Price Changes for {product_name}", fontsize=16)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Price", fontsize=12)
    plt.legend(loc='best')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid(True)

    # Show the plot
    plt.show()

# Plot the time series for each product
for product in marcadf['Descripcion'].unique():
    product_df = marcadf[marcadf['Descripcion'] == product]
    plot_product_price(product_df, product)

In [ ]:
# Define the start and end dates for filtering
start_date = '2023-08-01'
end_date = '2023-09-01'

# Filter the dataframe for the specified date range
filtered_df = marcadf[(marcadf['fecha'] >= start_date) & (marcadf['fecha'] <= end_date)]

# Display the filtered data
filtered_df

## 2.1. Eichenbum regular price

In [ ]:
# drop last exercise columns
marcadf = marcadf.drop(["Price Type", "Price Change Type"], axis = 1)

In [ ]:
def calculate_regular_price_by_month(df):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'YearMonth' column for grouping
    df['YearMonth'] = df['fecha'].dt.to_period('M')
    
    # Group by product description and YearMonth, then calculate mode for each group
    def mode_price(group):
        return group.mode()[0] if not group.mode().empty else None

    # Apply the mode calculation for each product and month
    regular_prices = df.groupby(['Descripcion', 'YearMonth'])['precio'].apply(mode_price).reset_index()
    
    # Merge regular price back into the original dataframe
    df = df.merge(regular_prices, on=['Descripcion', 'YearMonth'], how='left', suffixes=('', '_regular'))
    
    # Rename the new column for clarity
    df.rename(columns={'precio_regular': 'Regular_Price'}, inplace=True)
    
    # Drop the 'YearMonth' column as it's no longer needed
    df.drop(columns='YearMonth', inplace=True)
    
    return df

In [ ]:
marcadf = calculate_regular_price_by_month(marcadf)

In [ ]:
# Define the 'Price type' column based on comparison with Regular_Price
marcadf['Price_type'] = marcadf.apply(lambda row: 'Regular price' if row['precio'] == row['Regular_Price'] 
                            else 'Sales price' if row['precio'] < row['Regular_Price']
                            else 'Higher price', axis=1)

In [ ]:
def calculate_transition_matrix(df):
    # Define states based on whether price equals the regular price
    df['State'] = df.apply(lambda row: 1 if row['precio'] == row['Regular_Price'] else 2, axis=1)
    
    # Sort by Descripcion and fecha to ensure correct time ordering
    df = df.sort_values(by=['Descripcion', 'fecha'])
    
    # Create columns for previous state (shifted by one row)
    df['Prev_State'] = df.groupby('Descripcion')['State'].shift(1)
    
    # Drop rows where there is no previous state (i.e., first observation for each product)
    df = df.dropna(subset=['Prev_State'])
    
    # Create a transition table that counts the occurrences of each transition
    transition_counts = pd.crosstab(df['Prev_State'], df['State'])
    
    # Calculate the transition matrix as probabilities by dividing by row sums
    transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)
    
    return transition_matrix

#### Transition matrix 

Define a filter to erase missing products

In [ ]:
calculate_transition_matrix(marcadf)

#### Other statistics

In [ ]:
def calculate_price_statistics(df):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'YearMonth' column for monthly grouping
    df['YearMonth'] = df['fecha'].dt.to_period('M')
    
    # 1. Fraction of days spent at reference prices
    fraction_at_reference = (df['Price_type'] == 'Regular price').mean()
    
    # 2. Fraction of days spent below the reference price (Sales Price)
    fraction_nonreference_above = (df['Price_type'] == 'Sales price').sum()/(df['Price_type'] != 'Regular price').sum()
    
    # 3. Fraction of months in which daily prices are equal for the whole month
    def is_price_constant(group):
        return group['precio'].nunique() == 1  # True if all prices are the same

    price_constant_by_month = df.groupby(['Descripcion', 'YearMonth']).apply(is_price_constant)
    fraction_constant_months = price_constant_by_month.mean()
    
    # 4. Fraction of price changes that are from a non-reference price to a reference price
    df = df.sort_values(by=['Descripcion', 'fecha'])
    
    # Define state: 1 if 'Regular price', 0 if otherwise
    df['Is_Reference'] = (df['Price_type'] == 'Regular price').astype(int)
    
    # Calculate transitions within each product
    df['Prev_Is_Reference'] = df.groupby('Descripcion')['Is_Reference'].shift(1)
    
    # Count transitions from non-reference to reference price (from 0 to 1)
    non_ref_to_ref_changes = ((df['Prev_Is_Reference'] == 0) & (df['Is_Reference'] == 1)).sum()
    total_price_changes = ((df['precio'] != df['precio'].shift(1)).sum() - 1)
    
    fraction_non_ref_to_ref = non_ref_to_ref_changes / total_price_changes if total_price_changes > 0 else 0
    
    # Create a summary table with the results
    summary_table = pd.DataFrame({
        'Fraction of days at reference prices': [round(fraction_at_reference, 2)],
        'Fraction of non reference price below reference prices (Sales)': [round(fraction_nonreference_above, 2)],
        'Fraction of months with constant daily prices': [round(fraction_constant_months, 2)],
        'Fraction of price changes from non-reference to reference': [round(fraction_non_ref_to_ref, 2)]
    })
    
    return summary_table

In [ ]:
calculate_price_statistics(marcadf)

In [ ]:
def calculate_price_statistics_for_product(df):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'YearMonth' column for monthly grouping
    df['YearMonth'] = df['fecha'].dt.to_period('M')
    
    # 1. Fraction of days spent at reference prices
    fraction_at_reference = (df['Price_type'] == 'Regular price').mean()
    
    # 2. Fraction of days spent below the reference price (Sales Price)
    fraction_nonreference_above = ((df['Price_type'] == 'Sales price').sum())/((df['Price_type'] != 'Regular price').sum())
    
    # 3. Fraction of months in which daily prices are equal for the whole month
    def is_price_constant(group):
        return group['precio'].nunique() == 1  # True if all prices are the same

    price_constant_by_month = df.groupby('YearMonth').apply(is_price_constant)
    fraction_constant_months = price_constant_by_month.mean()
    
    # 4. Fraction of price changes that are from a non-reference price to a reference price
    df = df.sort_values(by=['fecha'])
    
    # Define state: 1 if 'Regular price', 0 if otherwise
    df['Is_Reference'] = (df['Price_type'] == 'Regular price').astype(int)
    
    # Calculate transitions within the product
    df['Prev_Is_Reference'] = df['Is_Reference'].shift(1)
    
    # Count transitions from non-reference to reference price (from 0 to 1)
    non_ref_to_ref_changes = ((df['Prev_Is_Reference'] == 0) & (df['Is_Reference'] == 1) & (df['precio'] != df['precio'].shift(1))).sum()
    total_price_changes = ((df['precio'] != df['precio'].shift(1)).sum() - 1)
    
    fraction_non_ref_to_ref = non_ref_to_ref_changes / total_price_changes if total_price_changes > 0 else 0
    
    # Return statistics as a Series
    return pd.Series({
        'Fraction of days at reference prices': round(fraction_at_reference, 2),
        'Fraction of non reference price below reference prices (Sales)': round(fraction_nonreference_above, 2),
        'Fraction of months with constant daily prices': round(fraction_constant_months, 2),
        'Fraction of price changes from non-reference to reference': round(fraction_non_ref_to_ref, 2)
    })


Verificar los errores 

In [ ]:
result1 = marcadf.groupby('Descripcion').apply(calculate_price_statistics_for_product)

In [ ]:
result1

In [ ]:
pruebadf = marcadf[marcadf["Descripcion"].str.contains("estrella galicia 500 ml")]

In [ ]:
def function_test(df):
    # Fraction of days at reference prices
    fraction_at_reference = round((df["precio"]==df["Regular_Price"]).mean(),2)
    # Fraction of non reference price below reference prices (Sales)
    fraction_nonreference_above = round((df[(df["Price_type"]!="Regular price")]["precio"]<df[(df["Price_type"]!="Regular price")]["Regular_Price"]).mean(),2)
    # Fraction of months with constant daily prices
    counter = 0
    for m in df["YearMonth"].unique():
        temp = df[df["YearMonth"]==m]
        if temp["precio"].nunique() == 1:
            counter += 1
    fraction_constant_months = round(counter/len(df["YearMonth"].unique()), 2)    
    # Fraction of price changes from non-reference to reference
    count_changes = 0
    count_to_reference = 0
    for p in range(len(df)):
        df = df.reset_index(drop = True)
        actual_price = df["precio"][p]
        if p > 0:
            past_price = df["precio"][(p-1)]
            cond1 = past_price != actual_price
            cond2 = df["Price_type"][p]=="Regular price"
            cond3 = df["Price_type"][(p-1)]!="Regular price"
            if (cond1):
                count_changes +=1
            if (cond1) & (cond2) & (cond3):
                count_to_reference +=1
    if count_changes > 0:
        fraction_non_ref_to_ref = round(count_to_reference/count_changes, 2)
    else:
        fraction_non_ref_to_ref = 0
    # Return statistics as a Series
    return pd.Series({
        'Fraction of days at reference prices': fraction_at_reference,
        'Fraction of non reference price below reference prices (Sales)': fraction_nonreference_above,
        'Fraction of months with constant daily prices': fraction_constant_months,
        'Fraction of price changes from non-reference to reference': fraction_non_ref_to_ref
    })

In [ ]:
result2 = marcadf.groupby('Descripcion').apply(function_test)

In [ ]:
# Add prefixes to the columns (except for the key column used for merging)
result1_prefixed = result1.add_suffix('_1')
result2_prefixed = result2.add_suffix('_2')

# Rename back the 'ID' column to avoid prefixing it in both DataFrames
result1_prefixed = result1_prefixed.rename(columns={'Descripcion_1': 'Descripcion'})
result2_prefixed = result2_prefixed.rename(columns={'Descripcion_2': 'Descripcion'})

# Merge the two DataFrames on the 'ID' column
merged_df = pd.merge(result1_prefixed, result2_prefixed, on='Descripcion', how='outer')
merged_result = merged_df.reindex(sorted(merged_df.columns), axis=1)
merged_result

In [ ]:
# 4. Fraction of price changes that are from a non-reference price to a reference price
df = df.sort_values(by=['fecha'])

# Define state: 1 if 'Regular price', 0 if otherwise
df['Is_Reference'] = (df['Price_type'] == 'Regular price').astype(int)

# Calculate transitions within the product
df['Prev_Is_Reference'] = df['Is_Reference'].shift(1)

# Count transitions from non-reference to reference price (from 0 to 1)
non_ref_to_ref_changes = ((df['Prev_Is_Reference'] == 0) & (df['Is_Reference'] == 1) & (df['precio'] != df['precio'].shift(1))).sum()
total_price_changes = ((df['precio'] != df['precio'].shift(1)).sum() - 1)

fraction_non_ref_to_ref = non_ref_to_ref_changes / total_price_changes if total_price_changes > 0 else 0

Pay atention to 2023-12-01 change (??)

In [ ]:
df = marcadf[marcadf["Descripcion"]=="cerv. lata pack 6 und aguila light 1614 ml"]

# 4. Fraction of price changes that are from a non-reference price to a reference price
df = df.sort_values(by=['fecha'])

# Define state: 1 if 'Regular price', 0 if otherwise
df['Is_Reference'] = (df['Price_type'] == 'Regular price').astype(int)

# Calculate transitions within the product
df['Prev_Is_Reference'] = df['Is_Reference'].shift(1)

# Count transitions from non-reference to reference price (from 0 to 1)
df["Price_change"] = (df['Prev_Is_Reference'] == 0) & (df['Is_Reference'] == 1)

df